# How to load SEC financial statements into pandas

Pull quarterly income statement, balance sheet and cash flow data from XBRL filings into a pandas DataFrame, and screen every filer on one metric.

Read it with charts at https://www.quantisnow.com/sec-api/guides/financial-statements-pandas. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

XBRL in 10-K and 10-Q filings has every number in the statements, but each company tags them its own way: one reports `Revenues`, another `RevenueFromContractWithCustomerExcludingAssessedTax`, a bank something else again. The financials endpoint maps them to the same field names, so one company's series and a screen across all of them are both one query.

## One company, eight quarters

### What were Apple's revenue and net income in each of the last eight quarters?

A quarterly series for a model or a chart, already split into quarters.

```shell
curl -X POST https://www.quantisnow.com/api/v1/financials \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:AAPL AND fiscalPeriod:(Q1 OR Q2 OR Q3 OR Q4)",
  "from": 0,
  "size": 5,
  "sort": [{"periodEnd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 40):

```json
{
  "total": {
    "value": 40,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0000320193:2026:Q4",
      "cik": "0000320193",
      "ticker": "AAPL",
      "companyName": "Apple Inc.",
      "fiscalYear": 2026,
      "fiscalPeriod": "Q4",
      "periodStart": "2026-06-28",
      "periodEnd": "2026-09-26",
      "filedOn": "2026-11-03",
      "form": "10-K",
      "accessionNo": "0000320193-26-000026",
      "sector": "Technology",
      "currency": "USD",
      "derived": true,
      "incomeStatement": {
        "revenue": 114477000000,
        "costOfRevenue": 59657000000,
        "grossProfit": 54820000000,
        "researchDevelopment": 11730000000,
        "sellingGeneralAdmin": 7459000000,
        "operatingIncome": 35631000000,
        "interestExpense": null,
        "pretaxIncome": 36135000000,
        "incomeTax": 5424000000,
        "netIncome": 30711000000,
        "epsBasic": 2.11,
        "epsDiluted": 2.11,
        "sharesDiluted": null
      },
      "balanceSheet": {
        "cash": 36305000000,
        "shortTermInvestments": 20417000000,
        "receivables": 43817000000,
        "inventory": 14541000000,
        "currentAssets": 173857000000,
        "goodwill": null,
        "totalAssets": 404548000000,
        "currentLiabilities": 169915000000,
        "shortTermDebt": 9906000000,
        "longTermDebt": 68208000000,
        "totalDebt": 78114000000,
        "totalLiabilities": 292455000000,
        "equity": 112093000000
      },
      "cashFlow": {
        "operatingCashFlow": 28101000000,
        "investingCashFlow": 1531000000,
        "financingCashFlow": -32871000000,
        "capex": 2242000000,
        "freeCashFlow": 25859000000,
        "dividendsPaid": 3943000000,
        "shareRepurchases": 24296000000,
        "stockCompensation": 3266000000,
        "depreciation": 3450000000
      }
    },
    {
      "id": "0000320193:2026:Q3",
      "cik": "0000320193",
      "ticker": "AAPL",
      "companyName": "Apple Inc.",
      "fiscalYear": 2026,
      "fiscalPeriod": "Q3",
      "periodStart": "2026-03-29",
      "periodEnd": "2026-06-27",
      "filedOn": "2026-07-31",
      "form": "10-Q",
      "accessionNo": "0000320193-26-000020",
      "sector": "Technology",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 109417000000,
        "costOfRevenue": 54647000000,
        "grossProfit": 54770000000,
        "researchDevelopment": 11729000000,
        "sellingGeneralAdmin": 7346000000,
        "operatingIncome": 35695000000,
        "interestExpense": null,
        "pretaxIncome": 36267000000,
        "incomeTax": 6478000000,
        "netIncome": 29789000000,
        "epsBasic": 2.03,
        "epsDiluted": 2.02,
        "sharesDiluted": 14714676000
      },
      "balanceSheet": {
        "cash": 39544000000,
        "shortTermInvestments": 22855000000,
        "receivables": 31398000000,
        "inventory": 11092000000,
        "currentAssets": 149818000000,
        "goodwill": null,
        "totalAssets": 383266000000,
        "currentLiabilities": 149326000000,
        "shortTermDebt": 11007000000,
        "longTermDebt": 71340000000,
        "totalDebt": 82347000000,
        "totalLiabilities": 275746000000,
        "equity": 107520000000
      },
      "cashFlow": {
        "operatingCashFlow": 34369000000,
        "investingCashFlow": -7757000000,
        "financingCashFlow": -32640000000,
        "capex": 2455000000,
        "freeCashFlow": 31914000000,
        "dividendsPaid": 4035000000,
        "shareRepurchases": 25105000000,
        "stockCompensation": 3401000000,
        "depreciation": 3320000000
      }
    }
  ]
}
```

In [ ]:
import os, requests
import pandas as pd

res = requests.post(
    "https://www.quantisnow.com/api/v1/financials",
    headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
    json={
        "query": "ticker:AAPL AND fiscalPeriod:(Q1 OR Q2 OR Q3 OR Q4)",
        "size": 8,
        "sort": [{"periodEnd": {"order": "desc"}}],
    },
)
df = pd.json_normalize(res.json()["data"]).set_index("periodEnd").sort_index()
print(df[["fiscalPeriod", "incomeStatement.revenue", "incomeStatement.netIncome", "cashFlow.freeCashFlow"]])

Companies don't file a fourth-quarter 10-Q. The Q4 rows are the annual 10-K minus the first three quarters, and carry `derived: true`.

## Screen every filer

### Which non-financial companies with over $10 billion in revenue had negative free cash flow last fiscal year?

A screen across every filer in one call, which needs the same field names for every company to work.

```shell
curl -X POST https://www.quantisnow.com/api/v1/financials \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "fiscalPeriod:FY AND fiscalYear:2025 AND incomeStatement.revenue:>10000000000 AND cashFlow.freeCashFlow:<0 AND currency:USD AND NOT sector:Finance",
  "from": 0,
  "size": 5,
  "sort": [{"incomeStatement.revenue": {"order": "desc"}}]
}'
```

Response (real, first 2 of 40):

```json
{
  "total": {
    "value": 40,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0000012927:2025:FY",
      "cik": "0000012927",
      "ticker": "BA",
      "companyName": "Boeing Company",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2025-01-01",
      "periodEnd": "2025-12-31",
      "filedOn": "2026-01-30",
      "form": "10-K",
      "accessionNo": "0001628280-26-004357",
      "sector": "Industrials",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 89463000000,
        "costOfRevenue": 85174000000,
        "grossProfit": 4289000000,
        "researchDevelopment": 3615000000,
        "sellingGeneralAdmin": 6090000000,
        "operatingIncome": 4281000000,
        "interestExpense": null,
        "pretaxIncome": 2635000000,
        "incomeTax": 397000000,
        "netIncome": 2235000000,
        "epsBasic": 2.49,
        "epsDiluted": 2.48,
        "sharesDiluted": 762300000
      },
      "balanceSheet": {
        "cash": 10921000000,
        "shortTermInvestments": 18479000000,
        "receivables": 2921000000,
        "inventory": null,
        "currentAssets": 128459000000,
        "goodwill": 17275000000,
        "totalAssets": 168235000000,
        "currentLiabilities": 108115000000,
        "shortTermDebt": 8461000000,
        "longTermDebt": 45637000000,
        "totalDebt": 54098000000,
        "totalLiabilities": 162778000000,
        "equity": 5454000000
      },
      "cashFlow": {
        "operatingCashFlow": 1065000000,
        "investingCashFlow": 499000000,
        "financingCashFlow": -3763000000,
        "capex": 2942000000,
        "freeCashFlow": -1877000000,
        "dividendsPaid": 331000000,
        "shareRepurchases": null,
        "stockCompensation": 426000000,
        "depreciation": 1953000000
      }
    },
    {
      "id": "0001001085:2025:FY",
      "cik": "0001001085",
      "ticker": "BN",
      "companyName": "Brookfield Corporation",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2025-01-01",
      "periodEnd": "2025-12-31",
      "filedOn": "2026-03-18",
      "form": "40-F",
      "accessionNo": "0001001085-26-000006",
      "sector": "Real Estate",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 75100000000,
        "costOfRevenue": 28351000000,
        "grossProfit": 46749000000,
        "researchDevelopment": null,
        "sellingGeneralAdmin": 5131000000,
        "operatingIncome": null,
        "interestExpense": null,
        "pretaxIncome": null,
        "incomeTax": 1135000000,
        "netIncome": 1307000000,
        "epsBasic": 0.51,
        "epsDiluted": 0.49,
        "sharesDiluted": 2367800000
      },
      "balanceSheet": {
        "cash": 16242000000,
        "shortTermInvestments": null,
        "receivables": 22772000000,
        "inventory": 5923000000,
        "currentAssets": null,
        "goodwill": 43355000000,
        "totalAssets": 518971000000,
        "currentLiabilities": null,
        "shortTermDebt": null,
        "longTermDebt": 13800000000,
        "totalDebt": 13800000000,
        "totalLiabilities": 352777000000,
        "equity": 166194000000
      },
      "cashFlow": {
        "operatingCashFlow": 10958000000,
        "investingCashFlow": -31777000000,
        "financingCashFlow": 21687000000,
        "capex": 14752000000,
        "freeCashFlow": -3794000000,
        "dividendsPaid": null,
        "shareRepurchases": null,
        "stockCompensation": null,
        "depreciation": 10379000000
      }
    }
  ]
}
```

Range queries work on any number field: `incomeStatement.revenue:>10000000000`, `balanceSheet.totalDebt:[0 TO 1000000]`, `incomeStatement.epsDiluted:<0`.